<img src="../assets/logo.png"/> <br>
# A full business solution our first project

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AIz') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=api_key
)

API key looks good so far


In [3]:
links = fetch_website_links("https://huggingface.com")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/tencent/Hy-MT2-1.8B',
 '/bytedance-research/Lance',
 '/NemoStation/Marlin-2B',
 '/tencent/Hy-MT2-30B-A3B',
 '/sapientinc/HRM-Text-1B',
 '/models',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview-2',
 '/spaces/victor/LongCat-Video-Avatar-1.5',
 '/spaces/HuggingFaceBio/carbon-demo',
 '/spaces/TencentARC/Pixal3D',
 '/spaces/cbensimon/wan2-2-fp8da-aoti-preview2',
 '/spaces',
 '/datasets/angrygiraffe/claude-opus-4.6-4.7-reasoning-8.7k',
 '/datasets/GD-ML/TransitLM',
 '/datasets/wikimedia/structured-wikipedia',
 '/datasets/TuringEnterprises/Open-MM-RL',
 '/dat

## First step: Have gemini-2.5-flash figure out which links are relevant

### Use a call to gemini-2.5-flash to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://huggingface.com"))


Here is the list of links on the website https://huggingface.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/tencent/Hy-MT2-1.8B
/bytedance-research/Lance
/NemoStation/Marlin-2B
/tencent/Hy-MT2-30B-A3B
/sapientinc/HRM-Text-1B
/models
/spaces/r3gm/wan2-2-fp8da-aoti-preview-2
/spaces/victor/LongCat-Video-Avatar-1.5
/spaces/HuggingFaceBio/carbon-demo
/spaces/TencentARC/Pixal3D
/spaces/cbensimon/wan2-2-fp8da-aoti-preview2
/spaces
/datasets/angrygiraffe/claude-opus-4.6-4.7-reasoning-8.7

In [12]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [13]:
select_relevant_links("https://huggingface.com")

{'links': [{'type': 'about page',
   'url': 'https://huggingface.com/huggingface'},
  {'type': 'brand information', 'url': 'https://huggingface.com/brand'},
  {'type': 'blog', 'url': 'https://huggingface.com/blog'},
  {'type': 'research papers', 'url': 'https://huggingface.com/papers'},
  {'type': 'enterprise solutions',
   'url': 'https://huggingface.com/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.com/pricing'},
  {'type': 'pro plan', 'url': 'https://huggingface.com/pro'},
  {'type': 'support page', 'url': 'https://huggingface.com/support'},
  {'type': 'product/service page', 'url': 'https://huggingface.com/models'},
  {'type': 'product/service page', 'url': 'https://huggingface.com/datasets'},
  {'type': 'product/service page', 'url': 'https://huggingface.com/spaces'},
  {'type': 'product/service page', 'url': 'https://huggingface.com/storage'},
  {'type': 'product/service page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'documentation', 'url': '

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://huggingface.com")

Selecting relevant links for https://huggingface.com by calling gemini-2.5-flash
Found 17 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.com/'},
  {'type': 'blog', 'url': 'https://huggingface.com/blog'},
  {'type': 'brand information', 'url': 'https://huggingface.com/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise solutions',
   'url': 'https://huggingface.com/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.com/pricing'},
  {'type': 'pro solutions', 'url': 'https://huggingface.com/pro'},
  {'type': 'support page', 'url': 'https://huggingface.com/support'},
  {'type': 'learn page', 'url': 'https://huggingface.com/learn'},
  {'type': 'documentation', 'url': 'https://huggingface.com/docs'},
  {'type': 'changelog', 'url': 'https://huggingface.com/changelog'},
  {'type': 'company GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'company Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'company LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'

## Second step: make the brochure!

Assemble all the details into another prompt to gemini-2.5-flash

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.com"))

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.com")

Selecting relevant links for https://huggingface.com by calling gemini-2.5-flash
Found 17 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ntencent/Hy-MT2-1.8B\nUpdated\nabout 3 hours ago\n•\n7.47k\n•\n978\nbytedance-research/Lance\nUpdated\nabout 9 hours ago\n•\n1.

In [20]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.com")

Selecting relevant links for https://huggingface.com by calling gemini-2.5-flash
Found 31 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the premier platform for the machine learning community to collaborate on cutting-edge AI. We are dedicated to democratizing good machine learning, one commit at a time.

## What We Offer:

*   **Models:** Explore and deploy over 2 million pre-trained models for a wide range of AI tasks.
*   **Datasets:** Access and contribute to a vast collection of over 500,000 datasets to train and evaluate your models.
*   **Spaces:** Discover and run over 1 million AI applications built by the community, showcasing the latest innovations in NLP, Computer Vision, Audio, and more.
*   **Buckets:** Secure and scalable storage solutions for your machine learning projects.
*   **HuggingChat:** Engage with advanced AI models for conversation and assistance.

## Our Mission & Culture:

At Hugging Face, we believe in the power of open-source collaboration to advance AI. Our culture is built around community, innovation, and accessibility. We foster an environment where individuals and organizations can freely share, discover, and build with AI.

## For Customers:

Hugging Face provides the infrastructure and tools necessary to accelerate your AI journey. Whether you're looking to leverage state-of-the-art models, access diverse datasets, or deploy your own AI applications, our platform offers a comprehensive solution. We also offer Enterprise solutions, including PRO subscriptions and dedicated Enterprise Support, to meet the needs of larger organizations.

## For Investors:

Hugging Face is at the forefront of the AI revolution, empowering a global community of developers and researchers. Our platform's rapid growth and the increasing demand for accessible AI tools position us for significant impact and scalability. We are building the foundational infrastructure for the future of AI.

## For Recruits:

Join a dynamic and passionate team that is shaping the future of artificial intelligence. At Hugging Face, you'll have the opportunity to work on challenging problems, collaborate with brilliant minds, and contribute to a mission-driven company. We are always looking for talented individuals to join us in democratizing machine learning.

**Current Openings:** Explore our careers page for exciting opportunities to be part of the Hugging Face team.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype.</span>
        </td>
    </tr>
</table>